# COHORT_DEFINITION — Success vs Failure Groups\n\n**Input:** `../data/berlin_marathon_clean.parquet`  \n**Output:** `../data/cohort_success.csv`, `../data/cohort_failure.csv`\n\n**Design:** Natural experiment — both groups pass the half-marathon at the same pace ("Golden Window"), but diverge in the second half. This isolates pacing execution as the variable.\n\n- **Success:** finish 2:58:00–2:59:59 (barrier breakers at their limit)\n- **Golden Window:** Success group mean HM ±30s\n- **Failure:** HM in Golden Window + finish >3:05:00 (metabolic collapse)\n- **Grey zone:** 3:00:00–3:04:59 excluded (ambiguous outcome)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/berlin_marathon_clean.parquet')
print(f"Clean dataset: {len(df):,} runners")
print(f"Columns: {list(df.columns)}")

In [ ]:
# --- 1. Define Success group ---
# Finish time 2:58:00–2:59:59 (barrier breakers at their physiological limit)
SUCCESS_MIN = 2 * 3600 + 58 * 60  # 10680s
SUCCESS_MAX = 2 * 3600 + 59 * 60 + 59  # 10799s

success = df[(df['net_time_sec'] >= SUCCESS_MIN) & (df['net_time_sec'] <= SUCCESS_MAX)].copy()
print(f"Success group (2:58:00–2:59:59): n = {len(success):,}")

# --- 2. Compute Golden Window ---
hm_mean = success['half_sec'].mean()
hm_std = success['half_sec'].std()
WINDOW_HALF = 30  # ±30 seconds

golden_low = hm_mean - WINDOW_HALF
golden_high = hm_mean + WINDOW_HALF

def sec_to_hms(s):
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = int(s % 60)
    return f"{h}:{m:02d}:{sec:02d}"

print(f"\nSuccess HM — mean: {sec_to_hms(hm_mean)} ({hm_mean:.1f}s), SD: {hm_std:.1f}s")
print(f"Golden Window: {sec_to_hms(golden_low)} – {sec_to_hms(golden_high)}")
print(f"  ({golden_low:.0f}s – {golden_high:.0f}s)")

## Validation: HM Distribution of Success Group

In [ ]:
# --- 3. Validate HM distribution of Success group ---
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
hm_min_vals = success['half_sec'] / 60  # convert to minutes
ax.hist(hm_min_vals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(hm_mean / 60, color='red', ls='--', label=f'Mean: {sec_to_hms(hm_mean)}')
ax.axvline(golden_low / 60, color='orange', ls=':', label=f'Window: ±{WINDOW_HALF}s')
ax.axvline(golden_high / 60, color='orange', ls=':')
ax.set_xlabel('Half-Marathon Time (minutes)')
ax.set_ylabel('Count')
ax.set_title('HM Distribution — Success Group (2:58–2:59 finishers)')
ax.legend()
plt.tight_layout()
plt.show()

# Detailed stats
print(f"HM stats for Success group (n={len(success):,}):")
print(f"  Mean:   {sec_to_hms(hm_mean)} ({hm_mean:.1f}s)")
print(f"  SD:     {hm_std:.1f}s ({hm_std/60:.2f} min)")
print(f"  Median: {sec_to_hms(success['half_sec'].median())}")
print(f"  IQR:    {sec_to_hms(success['half_sec'].quantile(0.25))} – {sec_to_hms(success['half_sec'].quantile(0.75))}")
print(f"  Range:  {sec_to_hms(success['half_sec'].min())} – {sec_to_hms(success['half_sec'].max())}")
print(f"\n  % within Golden Window: {(success['half_sec'].between(golden_low, golden_high)).mean()*100:.1f}%")

In [ ]:
# --- 4. Define Failure group ---
# All runners who passed HM within the Golden Window
in_window = df[(df['half_sec'] >= golden_low) & (df['half_sec'] <= golden_high)].copy()
print(f"Runners with HM in Golden Window: {len(in_window):,}")

# Failure: finish > 3:05:00 (clear metabolic collapse)
FAILURE_MIN = 3 * 3600 + 5 * 60  # 11100s

failure = in_window[in_window['net_time_sec'] > FAILURE_MIN].copy()
print(f"Failure group (finish > 3:05:00): n = {len(failure):,}")

# Grey zone: 3:00:00–3:04:59
GREY_MIN = 3 * 3600          # 10800s
grey = in_window[(in_window['net_time_sec'] >= GREY_MIN) & (in_window['net_time_sec'] <= FAILURE_MIN)].copy()
print(f"Grey zone (3:00:00–3:05:00): n = {len(grey):,} (excluded)")

# Also count those who broke sub-3h from the window
sub3_from_window = in_window[in_window['net_time_sec'] < GREY_MIN]
print(f"Sub-3h from window: n = {len(sub3_from_window):,}")

print(f"\n--- Sensitivity analysis: alternative Failure cutoffs ---")
for cutoff_min in [3, 5, 7, 10]:
    cutoff_sec = 3 * 3600 + cutoff_min * 60
    n = len(in_window[in_window['net_time_sec'] > cutoff_sec])
    print(f"  > 3:{cutoff_min:02d}:00 → n = {n:,}")

In [ ]:
# --- 5. Finish time distribution for runners in the Golden Window ---
fig, ax = plt.subplots(figsize=(10, 4))
finish_min = in_window['net_time_sec'] / 60
ax.hist(finish_min, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(SUCCESS_MIN / 60, color='green', ls='--', lw=1.5, label='Success zone start (2:58)')
ax.axvline(SUCCESS_MAX / 60, color='green', ls='--', lw=1.5, label='Success zone end (2:59:59)')
ax.axvline(GREY_MIN / 60, color='orange', ls=':', lw=1.5, label='Grey zone start (3:00)')
ax.axvline(FAILURE_MIN / 60, color='red', ls='--', lw=1.5, label='Failure cutoff (3:05)')
ax.set_xlabel('Finish Time (minutes)')
ax.set_ylabel('Count')
ax.set_title('Finish Time Distribution — Runners in the Golden Window')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6. Demographics summary ---
# Note: Success group is from the full dataset (2:58-2:59 finishers), NOT filtered by Golden Window
# Failure group is from Golden Window + finish > 3:05

print("=" * 60)
print(f"FINAL COHORT SUMMARY")
print("=" * 60)
print(f"Success: n = {len(success):,} (finish 2:58:00–2:59:59)")
print(f"Failure: n = {len(failure):,} (HM in Golden Window, finish > 3:05:00)")
print(f"Total:   n = {len(success) + len(failure):,}")
print()

for label, group in [('Success', success), ('Failure', failure)]:
    print(f"\n--- {label} (n={len(group):,}) ---")
    gender_pct = group['gender'].value_counts(normalize=True) * 100
    print(f"  Gender: M {gender_pct.get('M', 0):.1f}%, F {gender_pct.get('F', 0):.1f}%")
    print(f"  HM:     {sec_to_hms(group['half_sec'].mean())} ± {group['half_sec'].std():.1f}s")
    print(f"  Finish: {sec_to_hms(group['net_time_sec'].mean())} ± {group['net_time_sec'].std():.1f}s")
    print(f"  CV:     {group['pacing_cv'].mean():.2f}% ± {group['pacing_cv'].std():.2f}%")
    yr = group['year'].value_counts().sort_index()
    print(f"  Years:  {yr.index.min()}–{yr.index.max()} (peak: {yr.idxmax()} with n={yr.max()})")

In [ ]:
# --- 7. Save cohorts ---
success.to_csv('../data/cohort_success.csv', index=False)
failure.to_csv('../data/cohort_failure.csv', index=False)

print(f"Saved: ../data/cohort_success.csv ({len(success):,} rows)")
print(f"Saved: ../data/cohort_failure.csv ({len(failure):,} rows)")